In [6]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

np.random.seed(42)

In [9]:
data_dir = Path.cwd().parent /"data"/"processed"/"submissions"
train = pd.read_csv("../data/raw/train.csv")
test = pd.read_csv("../data/raw/test.csv")
sample = pd.read_csv(data_dir/"submission.csv")


In [11]:
THRESHOLD = 0.58
feat_cols = [c for c in train.columns if c not in ["id", "anomaly"]]

for df in [train, test]:
    df["channel"] = df["channel"].astype("category")
all_channels = pd.concat([train["channel"], test["channel"]]).astype("category").cat.categories
train["channel"] = train["channel"].cat.set_categories(all_channels)
test["channel"] = test["channel"].cat.set_categories(all_channels)

X = train[feat_cols]
y = train["anomaly"]
X_test = test[feat_cols]

model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.03, num_leaves=15,
    min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
    class_weight="balanced", random_state=42, verbosity=-1,
)
model.fit(X, y, categorical_feature=["channel"])

test_probs = model.predict_proba(X_test)[:, 1]
test_preds = (test_probs > THRESHOLD).astype(int)

submission = test[["id"]].copy()
submission["anomaly"] = test_preds
submission = sample[["id"]].merge(submission, on="id", how="left")

assert submission.shape[0] == 425
assert submission["anomaly"].isna().sum() == 0
assert set(submission["id"]) == set(sample["id"])

submission.to_csv(data_dir/"submission.csv", index=False)
print(submission.head())
print(f"\nPredicted anomaly rate: {submission.anomaly.mean():.4f} (train rate was 0.2044)")
print("Saved to submissions/submission.csv")

   id  anomaly
0   1        1
1   6        0
2  19        1
3  22        1
4  27        0

Predicted anomaly rate: 0.2024 (train rate was 0.2044)
Saved to submissions/submission.csv
